In [1]:
import os
import cv2
import json
# from io import BytesIO
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import scipy
import cv2
from skimage.measure import regionprops, label, shannon_entropy
from skimage.transform import resize
from skimage.color import rgb2gray
from skimage import img_as_ubyte
from brisque import BRISQUE
from skvideo.measure import niqe
from pypiqe import piqe
from mahotas.features import zernike_moments
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import transforms
import timm
import segmentation_models_pytorch as smp

# Patch imresize if missing
if not hasattr(scipy.misc, "imresize"):
    def imresize(arr, size, interp=None, mode=None):
        if isinstance(size, float):  # scale factor
            new_shape = (int(arr.shape[0] * size), int(arr.shape[1] * size))
        else:
            new_shape = size[:2]
        arr_resized = resize(arr, new_shape, order=3, mode="reflect", anti_aliasing=True)
        arr_resized = (arr_resized * 255).astype(np.uint8)
        return arr_resized
    scipy.misc.imresize = imresize

# Patch for deprecated NumPy aliases (for backward compatibility)
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool

In [2]:
# =========================
# CONFIG
# =========================
DATA_DIR = "/home/c/choton/beemachine/datasets/Others/fish-vista"
CLASSIFIER_NAME = "seresnext101_32x4d.gluon_in1k"
SEGMENTATION_NAME = "deeplabv3plus"
SEGMENTATION_ENCODER = "resnext50_32x4d"
SEGMENTATION_WEIGHTS = "/home/c/choton/beemachine/codes/AG_vision_2026/2_segmentation/FishVista/deeplabv3plus/lightning_logs/version_3/checkpoints/epoch=199-step=40600.ckpt"

MODEL = "arch2_fish_se_resnext101"
NUM_PARTS = 10
DEVICE_ID = 6
BATCH_SIZE = 32
IMAGE_SIZE = 224
EPOCHS = 100
LR = 1e-4
OPTIMIZER = "AdamW"
LOG_DIR = f"./{MODEL}_logs/"

device = torch.device(f"cuda:{DEVICE_ID}")

In [3]:
# Create SIFT and ORB detectors once
sift = cv2.SIFT_create()
orb = cv2.ORB_create()
bri_obj = BRISQUE(url=False)

def extract_base_features(mask):
    """Compute geometric, Zernike, Fourier, and texture shape descriptors from a binary mask."""
    
    features = ["area", "perimeter", "aspect_ratio", "extent", "solidity", "eccentricity", 
        "orientation", "circularity", "elongation", "compactness"]
    
    if mask is None or mask.sum() == 0:
        return {f: 0 for f in features}

    # --- Region properties ---
    # mask = mask.astype(np.uint8)
    labeled = label(mask)
    props = regionprops(labeled)
    if len(props) == 0:
        return {f: 0 for f in features}
    p = props[0]
    major_axis = p.major_axis_length
    minor_axis = p.minor_axis_length

    # ----- base shape features -----
    area = p.area
    perimeter = max(p.perimeter, 1e-6) # Ignoring too small perimeters
    aspect_ratio = major_axis / minor_axis if minor_axis > 0 else 0 # L_major / L_minor
    extent = p.extent
    solidity = p.solidity
    eccentricity = p.eccentricity
    orientation = p.orientation
    circularity = 4 * np.pi * area / (perimeter ** 2)
    elongation = 1 - (minor_axis / major_axis) if major_axis > 0 else 0
    # convexity = p.perimeter_convex / perimeter
    compactness = (perimeter ** 2) / (4 * np.pi * area + 1e-6)

    # ----- Assemble features -----
    features_d = {
        "area": area,
        "perimeter": perimeter,
        "aspect_ratio": aspect_ratio,
        "extent": extent,
        "solidity": solidity,
        "eccentricity": eccentricity,
        "orientation": orientation,
        "circularity": circularity,
        "elongation": elongation,
        "compactness": compactness
    }
    return features_d

def compute_sift_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY) # converts image into uint8 and mask as input
    keypoints, descriptors = sift.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 128), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_orb_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY)
    keypoints, descriptors = orb.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 32), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_hu_moments(mask):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    moments = cv2.moments(mask)
    hu = cv2.HuMoments(moments).flatten()
    hu = np.log(np.abs(hu) + 1e-12) # log-scale for stability
    return hu

def compute_zernike_moments(mask, degree=8):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    radius = min(mask.shape) // 2
    mask_norm = mask / mask.max() if mask.max() > 0 else mask
    zern = zernike_moments(mask_norm, radius=radius, degree=degree)
    return zern

# *** Updated fourier descriptors (Dec 4, 2025)
def compute_fourier_descriptors(mask, image=None, fourier_harmonics=20, visualize=False):
    if not isinstance(mask, np.ndarray): # Ensure proper mask format
        mask = mask.numpy().astype(np.uint8)
    # --- 2. Find largest contour (object part) ---
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) < 3:
        # Too few points for Fourier transform
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    
    # Translation invariance: center contour
    complex_contour = cnt[:,0,0] + 1j * cnt[:,0,1]
    fd = np.fft.fft(complex_contour)
    
    if visualize: # ** IMPORTANT: Visualization uses raw contour (so you can see the real shape), descriptors are centered.
        # Convert image if needed
        H, W = mask.shape
        if image is not None:
            if isinstance(image, torch.Tensor):
                image = image.detach().cpu().numpy().transpose(1, 2, 0)
            elif isinstance(image, Image.Image):
                image = np.array(image.convert('RGB'))
            elif image.dtype != np.uint8:  # NumPy float → uint8
                image = (image*255).astype(np.uint8)
            img_draw = image.copy()
        else:
            img_draw = np.zeros((H, W, 3), dtype=np.uint8)
        cv2.drawContours(img_draw, [cnt.astype(np.int32)], -1, (0, 255, 0), 2)

        fd_recon = fd.copy()
        keep = fourier_harmonics
        if 2 * keep < len(fd_recon):
            fd_recon[keep:-keep] = 0 # Safe zeroing
        else:
            fd_recon[keep:] = 0
        recon = np.fft.ifft(fd_recon)
        pts = np.column_stack((recon.real, recon.imag)).astype(np.int32)

        for i in range(len(pts)):
            cv2.line(img_draw, tuple(pts[i]), tuple(pts[(i + 1) % len(pts)]), (255, 0, 255), 1)
        plt.figure(figsize=(16, 6))
        plt.imshow(img_draw)
        plt.axis('off')
        plt.title("Shape Descriptors Overlay")
        plt.legend(
            handles=[
                Patch(facecolor='green', edgecolor='green'),
                Patch(facecolor='magenta', edgecolor='magenta')
            ],
            labels=["Contour", "Fourier Reconstruction"],
            loc='upper right'
        )
        plt.show()
    
    cnt_centered = complex_contour - np.mean(complex_contour)
    fd = np.fft.fft(cnt_centered)
    if len(fd) < 2 or np.abs(fd[1]) == 0:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)

    # Scale invariance: divide by first descriptor magnitude
    fd = fd / np.abs(fd[1])

    # Rotation invariance: use only magnitudes
    fd_normalized = np.abs(fd)

    # Keep only first N harmonics
    fd_truncated = fd_normalized[:fourier_harmonics]
    if len(fd_truncated) < fourier_harmonics:
        fd_truncated = np.concatenate([fd_truncated, np.full((fourier_harmonics - len(fd_truncated)), np.nan)])
    return fd_truncated

def extract_shape_features(image, mask):
    # Compute base features
    features = extract_base_features(mask)

    # Compute sift features
    sift_kp, sift_ds = compute_sift_features(image, mask)
    sift_sizes = [k.size for k in sift_kp]
    if sift_ds.shape[0] > 0:
        sift_mean_ds = np.nanmean(sift_ds, axis=0)
    else:
        sift_mean_ds = np.full(128, np.nan)
    sift_dict = {f'sift_ds{i+1}': sift_mean_ds[i] for i in range(len(sift_mean_ds))}
    sift_dict['sift_kp_n'] = len(sift_kp)
    sift_dict['sift_kp_size'] = np.max(sift_sizes) if sift_sizes else 0

    # Compute orb features
    orb_kp, orb_ds = compute_orb_features(image, mask)
    if orb_ds.shape[0] > 0:
        orb_mean_ds = np.nanmean(orb_ds, axis=0)
    else:
        orb_mean_ds = np.full(32, np.nan)
    orb_dict = {f'orb_ds{i+1}': orb_mean_ds[i] for i in range(len(orb_mean_ds))}
    orb_dict['orb_kp_n'] = len(orb_kp)

    # Compute hu moments
    hu_moments = compute_hu_moments(mask)
    hu_dict = {f"hu{i+1}": hu_moments[i] for i in range(len(hu_moments))}

    # Compute Zernike moments
    zern_moments = compute_zernike_moments(mask, degree=8)
    zern_dict = {f"zernike_{i+1}": zern_moments[i] for i in range(len(zern_moments))}

    # Compute fourier descriptors
    fourier_ds = compute_fourier_descriptors(mask, fourier_harmonics=20)
    fourier_dict = {f"fourier_{i+1}": fourier_ds[i] for i in range(len(fourier_ds))}

    features.update(sift_dict)
    features.update(orb_dict)
    features.update(hu_dict)
    features.update(zern_dict)
    features.update(fourier_dict)
    converted = {k: np.float32(v) for k, v in features.items()}
    return converted

def extract_visual_features(image, mask):
    # --- 1. Ensure binary uint8 mask ---
    if not isinstance(mask, np.ndarray):
        mask = mask.numpy().astype(np.uint8)
    # Convert image to numpy
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    elif isinstance(image, Image.Image):
        image = np.array(image.convert('RGB'))
    img_cropped = np.zeros_like(image)
    img_cropped[mask==1] = image[mask==1]
    # plt.imshow(img_cropped)

    # --- Brightness ---
    brightness = np.mean(img_cropped)

    # --- Contrast (standard deviation of luminance) ---
    gray = rgb2gray(img_cropped)
    contrast = np.std(gray)

    # --- Sharpness (variance of Laplacian) ---
    gray_8u = (gray * 255).astype(np.uint8)
    lap_var = cv2.Laplacian(gray_8u, cv2.CV_64F).var()

    # --- Colorfulness (Hasler & Süsstrunk, 2003) ---
    (R, G, B) = cv2.split(img_cropped)
    rg = np.abs(R - G)
    yb = np.abs(0.5 * (R + G) - B)
    std_rg, std_yb = np.std(rg), np.std(yb)
    mean_rg, mean_yb = np.mean(rg), np.mean(yb)
    colorfulness = np.sqrt(std_rg**2 + std_yb**2) + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2)

    # --- Entropy (texture complexity) ---
    entropy = shannon_entropy(gray)

    # BRISQUE
    bri_obj = BRISQUE(url=False)
    brisque_score = bri_obj.score(img_cropped)

    # NIQE
    niqe_score = niqe(gray)

    # PIQE
    piqe_score, activityMask, noticeableArtifactMask, noiseMask = piqe(gray)

    # --- Aggregate descriptors ---
    descriptors = {
        "brightness": np.float32(brightness),
        "contrast": np.float32(contrast),
        "sharpness": np.float32(lap_var),
        "colorfulness": np.float32(colorfulness),
        "entropy": np.float32(entropy),
        "brisque": np.float32(brisque_score),
        "niqe": np.float32(niqe_score.item()),
        "piqe": np.float32(piqe_score)
    }
    return descriptors

def extract_combined_features(image, mask): 
    # ---- Convert once ----
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    if isinstance(image, Image.Image):
        image = np.asarray(image)
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy().astype(np.uint8)
    mask_u8 = mask.astype(np.uint8)
    image_f = (image - image.min()) / (image.max() - image.min() + 1e-8) # Linearly rescale to [0, 1] and avoid division by zero
    image_u8 = img_as_ubyte(image_f)

    combined_features = extract_shape_features(image_u8, mask_u8)
    vis_features = extract_visual_features(image_f, mask_u8)
    combined_features.update(vis_features)
    return combined_features

def extract_all_features(image, mask):
    abdomen_mask = mask == 1
    head_mask = mask == 2
    thorax_mask = mask == 3
    full_mask = mask > 0

    head_feats = extract_combined_features(image, head_mask)
    thorax_feats = extract_combined_features(image, thorax_mask)
    abdomen_feats = extract_combined_features(image, abdomen_mask)
    body_feats = extract_combined_features(image, full_mask)

    # Ratios
    area_sum = head_feats["area"] + thorax_feats["area"] + abdomen_feats["area"]
    ratios = {
        "head_to_thorax_area": head_feats["area"] / (thorax_feats["area"] + 1e-6),
        "thorax_to_abdomen_area": thorax_feats["area"] / (abdomen_feats["area"] + 1e-6),
        "head_to_total_area": head_feats["area"] / (area_sum + 1e-6),
        "thorax_to_total_area": thorax_feats["area"] / (area_sum + 1e-6),
        "abdomen_to_total_area": abdomen_feats["area"] / (area_sum + 1e-6),
    }

    record = {}
    record.update({f"head_{k}": v for k, v in head_feats.items()})
    record.update({f"thorax_{k}": v for k, v in thorax_feats.items()})
    record.update({f"abdomen_{k}": v for k, v in abdomen_feats.items()})
    record.update({f"full_{k}": v for k, v in body_feats.items()})
    record.update(ratios)
    return record

In [4]:
# Load classifier
def load_classifier(name, weights, num_classes, device_id):
    # Define the classification model and load weights
    model = timm.create_model(name, pretrained=False, num_classes=num_classes).to(f"cuda:{device_id}")
    state_dict_c = torch.load(weights, map_location=f"cuda:{device_id}")
    model.load_state_dict(state_dict_c) # Load the classifier state dict
    return model

# Load segmentation model
def load_segmentation(name, encoder_name, checkpoint_path):    
    # Define the segmentation model and load weights
    model = smp.create_model(
                name,
                encoder_name=encoder_name,
                in_channels = 3, # 3 for RGB images
                classes = NUM_PARTS
            ).to(f"cuda:{DEVICE_ID}")
    # checkpoint_path = r"/home/c/choton/beemachine/codes/everyday_test/oct1_25/segmentation_models/lightning_logs/fpn/lightning_logs/version_0/checkpoints/epoch=199-step=37400.ckpt"
    checkpoint = torch.load(checkpoint_path, map_location=rf"cuda:{DEVICE_ID}")

    # Sometimes Lightning adds "state_dict"
    if "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

        # Remove "model." or "net." prefixes if present
        new_state_dict = {k.replace("model.", "").replace("net.", ""): v 
                        for k, v in state_dict.items() if k not in ["mean", "std"]}

        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(checkpoint)
    # model.to(f"cuda:{DEVICE_ID}")
    return model

In [5]:
# Load the classification splits and check the shape
train_df = pd.read_csv(os.path.join(DATA_DIR, "classification_train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "classification_val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "classification_test.csv"))

classes = sorted(train_df['standardized_species'].unique())
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}
print(f'Shape of FishVista classification datasets,  train: {train_df.shape}, validation: {val_df.shape}, test): {test_df.shape}')
print(f'Columns of the test dataset:', list(test_df.columns))

/tmp/ipykernel_1592933/3138452075.py:2: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(os.path.join(DATA_DIR, "classification_train.csv"))


Shape of FishVista classification datasets,  train: (39800, 17), validation: (6779, 17), test): (9781, 17)
Columns of the test dataset: ['filename', 'source_filename', 'original_format', 'arkid', 'family', 'source', 'owner', 'standardized_species', 'original_url', 'license', 'adipose_fin', 'pelvic_fin', 'barbel', 'multiple_dorsal_fin', 'file_name', 'verbatim_species', 'species']


In [6]:
train_df.head()

,filename,source_filename,original_format,arkid,family,source,owner,standardized_species,original_url,license,adipose_fin,pelvic_fin,barbel,multiple_dorsal_fin,file_name,verbatim_species,species
0,INHS_FISH_107832.jpg,INHS_FISH_107832.jpg,jpg,79497k9r,Ictaluridae,GLIN,INHS,noturus nocturnus,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_1/INHS_FISH_107832.jpg,NaN,NaN
1,INHS_FISH_25132.jpg,INHS_FISH_25132.jpg,jpg,jw38b330,Cyprinidae,GLIN,INHS,notropis atherinoides,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_1/INHS_FISH_25132.jpg,NaN,NaN
2,INHS_FISH_40481.jpg,INHS_FISH_40481.jpg,jpg,zj534d84,Cyprinidae,GLIN,INHS,notropis texanus,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_1/INHS_FISH_40481.jpg,NaN,NaN
3,ark-_65665_m38f1c86b3282243d9b4b96d24f1983f3d.jpg,ark-_65665_m38f1c86b3282243d9b4b96d24f1983f3d.jpg,jpg,g898t46h,mullidae,iDigBio,usnm,parupeneus multifasciatus,https://fishair.org/hdr-share/ftp/ark/89609/iD...,Usage Conditions Apply,NaN,NaN,NaN,NaN,Images/chunk_1/ark-_65665_m38f1c86b3282243d9b4...,NaN,NaN
4,INHS_FISH_15401.jpg,INHS_FISH_15401.jpg,jpg,8c45003f,Poeciliidae,GLIN,INHS,gambusia affinis,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_1/INHS_FISH_15401.jpg,NaN,NaN


In [7]:
val_df.head()

,filename,source_filename,original_format,arkid,family,source,owner,standardized_species,original_url,license,adipose_fin,pelvic_fin,barbel,multiple_dorsal_fin,file_name,verbatim_species,species
0,INHS_FISH_22501.jpg,INHS_FISH_22501.jpg,jpg,6w839r77,Cyprinidae,GLIN,INHS,notropis blennius,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_5/INHS_FISH_22501.jpg,NaN,NaN
1,INHS_FISH_59332.jpg,INHS_FISH_59332.jpg,jpg,m587qt88,Centrarchidae,GLIN,INHS,lepomis gibbosus,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_5/INHS_FISH_59332.jpg,NaN,NaN
2,60898_lat_FMNH_FZ_l.jpg,60898_lat_FMNH_FZ_l.jpg,jpg,wt71491f,Cyprinidae,GLIN,FMNH,notropis stramineus,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_5/60898_lat_FMNH_FZ_l.jpg,NaN,NaN
3,INHS_FISH_33052.jpg,INHS_FISH_33052.jpg,jpg,q194mh91,Centrarchidae,GLIN,INHS,chaenobryttus gulosus,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_5/INHS_FISH_33052.jpg,NaN,NaN
4,UWZM-F-0001159.JPG,UWZM-F-0001159.JPG,jpg,gc45fw0v,Centrarchidae,GLIN,UWZM,lepomis gibbosus,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC0 1.0,NaN,NaN,NaN,NaN,Images/chunk_5/UWZM-F-0001159.JPG,NaN,NaN


In [8]:
test_df.head()

,filename,source_filename,original_format,arkid,family,source,owner,standardized_species,original_url,license,adipose_fin,pelvic_fin,barbel,multiple_dorsal_fin,file_name,verbatim_species,species
0,JFBM-FISH-0022553.jpg,JFBM-FISH-0022553.jpg,jpg,fm82032s,Ictaluridae,GLIN,JFBM,noturus gyrinus,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC0 1.0,NaN,NaN,NaN,NaN,Images/chunk_4/JFBM-FISH-0022553.jpg,NaN,NaN
1,ark-_65665_m3e9156fee2bb64a56b45bacbbab425d07.jpg,ark-_65665_m3e9156fee2bb64a56b45bacbbab425d07.jpg,jpg,9f40283k,apogonidae,iDigBio,usnm,fowleria vaiulae,https://fishair.org/hdr-share/ftp/ark/89609/iD...,Usage Conditions Apply,NaN,NaN,NaN,NaN,Images/chunk_4/ark-_65665_m3e9156fee2bb64a56b4...,NaN,NaN
2,INHS_FISH_61023.jpg,INHS_FISH_61023.jpg,jpg,0695rf52,Centrarchidae,GLIN,INHS,lepomis cyanellus,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_4/INHS_FISH_61023.jpg,NaN,NaN
3,80544_lat_FMNH_FZ#6.jpg,80544_lat_FMNH_FZ.jpg,jpg,9004rk30,Cyprinidae,GLIN,FMNH,cyprinella lutrensis,https://fishair.org/hdr-share/ftp/ark/89609/GL...,CC BY-NC,NaN,NaN,NaN,NaN,Images/chunk_4/80544_lat_FMNH_FZ#6.jpg,NaN,NaN
4,m3b534eebc-6ae8-4a4d-afb0-baa0fe87cfaa.jpg,m3b534eebc-6ae8-4a4d-afb0-baa0fe87cfaa.jpg,jpg,s838tx45,gobiidae,iDigBio,usnm,gobiidae,https://fishair.org/hdr-share/ftp/ark/89609/iD...,Usage Conditions Apply,NaN,NaN,NaN,NaN,Images/chunk_4/m3b534eebc-6ae8-4a4d-afb0-baa0f...,NaN,NaN


In [9]:
class FishVistaDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(DATA_DIR, "Images", row["filename"])
        image = Image.open(img_path).convert("RGB")

        label = class_to_idx[row["standardized_species"]]
        filename = row["filename"]

        image = self.transform(image)
        return image, label, filename

In [10]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    # transforms.RandomHorizontalFlip(),
    # transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = FishVistaDataset(train_df, train_transform)
val_dataset = FishVistaDataset(val_df, val_transform)
test_dataset = FishVistaDataset(test_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)

print(f"Classes: {num_classes} | Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Classes: 1758 | Train: 39800 | Val: 6779 | Test: 9781


In [11]:
# ======================================================
# Helper: Load shape features
# ======================================================
# def load_shape_features(csv_path):
#     df = pd.read_csv(csv_path).drop(columns=["image"])
#     df = df.fillna(df.mean())
#     # labels = torch.tensor(df["label"].values, dtype=torch.long)
#     # Drop non-feature columns: assuming CSV columns are ['image', 'label', ...features]
#     feats = df.to_numpy(dtype=np.float32)
#     feats_tensor = torch.tensor(feats, dtype=torch.float32)
#     return feats_tensor #, labels

def load_shape_df(csv_path):
    df = pd.read_csv(csv_path)
    filenames = df["image"].values
    feats = df.drop(columns=["image"])
    feats = feats.fillna(feats.mean())
    feats = feats.values.astype(np.float32)
    return filenames, feats

In [12]:
train_feats_path = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/FishVista/shape_features_fishvista/fishvista_shape_features_train.csv"
val_feats_path   = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/FishVista/shape_features_fishvista/fishvista_shape_features_val.csv"
test_feats_path  = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/FishVista/shape_features_fishvista/fishvista_shape_features_test.csv"

# Load raw
train_names, train_raw = load_shape_df(train_feats_path)
val_names, val_raw     = load_shape_df(val_feats_path)
test_names, test_raw   = load_shape_df(test_feats_path)

# Compute normalization from TRAIN only
mean = train_raw.mean(axis=0)
std  = train_raw.std(axis=0) + 1e-6

train_norm = (train_raw - mean) / std
val_norm   = (val_raw - mean) / std
test_norm  = (test_raw - mean) / std

# Convert to filename maps
train_shape_map = {
    name: torch.tensor(feat) for name, feat in zip(train_names, train_norm)
}
val_shape_map = {
    name: torch.tensor(feat) for name, feat in zip(val_names, val_norm)
}
test_shape_map = {
    name: torch.tensor(feat) for name, feat in zip(test_names, test_norm)
}

FEATURE_SIZE = train_norm.shape[1]
print(f"Train feats: {train_raw.shape}, Val feats: {val_raw.shape}, Test feats: {test_raw.shape}")
print(f"number of shape features:", FEATURE_SIZE)
# print(f"Train mean: {mean}, train std {std}")

Train feats: (39800, 2376), Val feats: (6779, 2376), Test feats: (9781, 2376)
number of shape features: 2376


In [13]:
class ShapeEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(128, embed_dim)

    def forward(self, mask_tensor):
        """
        mask_tensor: (B, 3, 64, 64)
        """
        feat = self.encoder(mask_tensor)
        feat = feat.flatten(1)
        z = self.fc(feat)
        return z  

class GatedFusion(nn.Module):
    def __init__(self, img_dim, shape_dim):
        super().__init__()

        self.shape_proj = nn.Linear(shape_dim, img_dim)

        self.gate = nn.Sequential(
            nn.Linear(img_dim + img_dim, img_dim),
            nn.LayerNorm(img_dim),
            nn.Sigmoid()
        )

    def forward(self, z_img, z_shape):
        z_shape_proj = self.shape_proj(z_shape)

        concat = torch.cat([z_img, z_shape_proj], dim=1)
        g = self.gate(concat)

        fused = z_img + g * (z_shape_proj - z_img)
        return fused

In [14]:
class DeepShapeFusionModel(nn.Module):
    def __init__(self, num_classes, shape_embed_dim=256):
        super().__init__()
        self.num_shape_feats = FEATURE_SIZE

        # Modules
        self.seg_module = load_segmentation(
            name=SEGMENTATION_NAME, 
            encoder_name=SEGMENTATION_ENCODER, 
            checkpoint_path=SEGMENTATION_WEIGHTS
        )
        self.backbone = timm.create_model(model_name=CLASSIFIER_NAME, pretrained=True, num_classes=0)
        self.shape_encoder = ShapeEncoder(shape_embed_dim)

        self.fusion = GatedFusion(
            img_dim=self.backbone.num_features,
            shape_dim=shape_embed_dim
        )

        total_feats = self.backbone.num_features #+ self.num_shape_feats # + shape_embed_dim

        self.classifier = nn.Sequential(
                                nn.LayerNorm(total_feats),
                                nn.Dropout(0.3),
                                nn.Linear(total_feats, num_classes)
                            )

        for p in self.seg_module.parameters():
            p.requires_grad = False
        self.seg_module.eval()

    def forward(self, x, shape_feats=None):

        # ---- Segmentation ----
        with torch.no_grad():
            mask_logits = self.seg_module(x)
            masks = torch.argmax(mask_logits, dim=1)  # (B, H, W) with labels 0,1,2,3
        # mask_logits = self.seg_module(x)
        # mask_probs = torch.softmax(mask_logits, dim=1)  # example single part

        # Extract part embeddings
        parts = mask_logits[:, 1:, :, :]  # remove background, will try mask_probs as well
        mask_binary = parts.sum(dim=1, keepdim=True)
        edge = torch.abs(
            torch.nn.functional.avg_pool2d(mask_binary, 3, stride=1, padding=1)
            - mask_binary
        )
        shape_tensor = torch.cat([mask_binary, edge, mask_binary], dim=1)
        z_shape = self.shape_encoder(shape_tensor)

        # ---- Extract hand-crafted shape descriptors ----
        # ---- Shape features ----
        if shape_feats is None:
            batch_size = x.size(0)
            batch_feats = []
            for i in range(batch_size):
                img_np = x[i].detach().cpu().numpy().transpose(1, 2, 0)
                mask_np = masks[i].detach().cpu().numpy()
                feats_dict = extract_all_features(img_np, mask_np)
                # Convert dict to tensor (relies on consistent insertion order for 937 features)
                feat_tensor = torch.tensor(list(feats_dict.values()), dtype=torch.float32)
                batch_feats.append(feat_tensor)
            shape_feats = torch.stack(batch_feats).to(x.device)  # (B, 937)
        # shape_feats = torch.tensor(shape_feats).to(x.device)        

        # ---- Image Encoding ----
        z_img = self.backbone(x)
        fused = self.fusion(z_img, z_shape)
        # combined = torch.cat([fused, shape_feats], dim=1)  # (B, total_feats)
        combined = fused # TESTING

        # ---- Classification ----
        out = self.classifier(combined)

        return out, mask_logits, z_shape


In [15]:
device = torch.device(f"cuda:{DEVICE_ID}")
model = DeepShapeFusionModel(num_classes=num_classes, shape_embed_dim=512)
model.to(device)

DeepShapeFusionModel(
  (seg_module): DeepLabV3Plus(
    (encoder): ResNetEncoder(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_runni

In [16]:
# Setting up the criterion, optimizer and scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  #label_smoothing=0.1)
if OPTIMIZER == "SGD":
    optimizer = torch.optim.SGD(
        model.parameters(), 
        lr=LR, 
        momentum=0.9, 
        weight_decay=1e-4)
elif OPTIMIZER == "AdamW":
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=LR, 
        weight_decay=1e-4)
else:
    optimizer = torch.optim.RMSprop(
        model.parameters(),
        lr=LR,       # typically around 0.256 / batch_size scaling, you can tune
        alpha=0.9,              # decay (smoothing constant)
        eps=0.001,              # numerical stability
        momentum=0.9,           # as in paper
        weight_decay=1e-5,      # L2 regularization
        centered=False          # same as paper’s setup
    )
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
scaler = torch.amp.GradScaler('cuda')

In [17]:
def train_one_epoch(model, train_loader, shape_map, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0, 0, 0
    pbar = tqdm(train_loader, desc="[Train]", position=0, leave=False)
    b_pos = 0 # Batch position
    for batch_idx, (images, labels, names) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)        
        # shape_feats_batch = train_feats[b_pos:b_pos+images.size(0)].to(device)
        # shape_feats_batch = train_feats[indices].to(device)
        shape_feats_batch = torch.stack([shape_map[n] for n in names]).to(device)        
        b_pos += images.size(0)

        optimizer.zero_grad()
        outputs, _, _ = model(images, shape_feats_batch)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def validate_one_epoch(model, val_loader, shape_map, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    b_pos = 0 # Batch position
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="[Val]", position=0, leave=False)
        for batch_idx, (images, labels, names) in enumerate(pbar):
            images, labels = images.to(device), labels.to(device)
            # shape_feats_batch = val_feats[b_pos:b_pos+images.size(0)].to(device)
            # shape_feats_batch = val_feats[indices].to(device)
            shape_feats_batch = torch.stack([shape_map[n] for n in names]).to(device)
            b_pos += images.size(0)

            # optimizer.zero_grad()
            outputs, _, _ = model(images, shape_feats_batch)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, train_shape_map, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, val_shape_map, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [18]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    log_dir=LOG_DIR
)

Starting the first epoch...


[Train]:   0%|          | 0/1244 [00:00<?, ?it/s]

 Epoch 1/100 | Train Loss: 3.0995 | Train Acc: 0.5932 | Val Loss: 3.0426 | Val Acc: 0.6004


 Epoch 2/100 | Train Loss: 2.0416 | Train Acc: 0.7866 | Val Loss: 2.5125 | Val Acc: 0.6802


 Epoch 3/100 | Train Loss: 1.6080 | Train Acc: 0.8853 | Val Loss: 2.2208 | Val Acc: 0.7420


...

 Epoch 27/100 | Train Loss: 1.0959 | Train Acc: 1.0000 | Val Loss: 2.0754 | Val Acc: 0.8286


 Epoch 28/100 | Train Loss: 1.0957 | Train Acc: 1.0000 | Val Loss: 2.0762 | Val Acc: 0.8287


 Epoch 29/100 | Train Loss: 1.0955 | Train Acc: 1.0000 | Val Loss: 2.0784 | Val Acc: 0.8280


 Epoch 30/100 | Train Loss: 1.0954 | Train Acc: 1.0000 | Val Loss: 2.0770 | Val Acc: 0.8277


KeyboardInterrupt: 

In [19]:
test_loss, test_acc = validate_one_epoch(model, test_loader, test_shape_map, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Test Loss: 1.9750 | Test Acc: 0.8436


In [ ]:
# Add right after model.eval() in validate_one_epoch
print("=== Running validation diagnostics ===")

with torch.no_grad():
    for i, (images, labels, names) in enumerate(val_loader):
        if i >= 3: break  # only first few batches
            
        images = images.to(device)
        labels = labels.to(device)
        shape_feats_batch = torch.stack([val_shape_map[n] for n in names]).to(device)
        
        outputs, _, _ = model(images, shape_feats_batch)
        
        loss = criterion(outputs, labels)
        
        probs = torch.softmax(outputs, dim=1)
        max_probs, preds = outputs.max(dim=1)
        
        print(f"Batch {i} | loss = {loss.item():.4f}")
        print(f"Mean max-prob = {max_probs.mean().item():.4f}")
        print(f"Acc = {(preds == labels).float().mean().item():.4f}")
        print(f"Logits range = {outputs.min().item():.2f} → {outputs.max().item():.2f}")
        print("-"*60)